![IITIS](pictures/logoIITISduze.png)

# Wykorzystanie procesorów graficznych (GPU) w algorytmach heurystycznych

wykorzystamy paczkę cupy

## Podstawy programowania GPU

In [1]:
# funkcje pomocniczne

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt




test_pegasus = os.path.join("instancje", "Pegasus", "P4_CBFM-P.txt")  # E = -469.0
full_pegasus = os.path.join("instancje", "Pegasus", "P16_CBFM-P.txt")  # E = -12772.0



def read_instance(path: os.PathLike):
    df = pd.read_csv(path, sep=" ", header=None, comment="#", names=["i", "j", "value"])

    n = max(df[["i", "j"]].max())
    h = np.zeros(n)
    J = np.zeros((n, n))
    
    for row in df.itertuples():
        if row.i == row.j:
            h[row.i - 1] = row.value
        elif row.i > row.j:
            J[row.j - 1, row.i - 1] = row.value  # by zachować górnotrójkątność
        else:
            J[row.i - 1, row.j - 1] = row.value
            
    return J, h


def dwave_conv_to_minus_half_convention(J: np.ndarray, h: np.ndarray):
    n = len(h)
    herminian_matrix = np.zeros((n, n))

    # de facto wyciągamy -1/2 przed macierz i zamieniamy ją na hermitowską
    for i in range(n):
        for j in range(i + 1, n):
            J_ij = J[i, j]
            herminian_matrix[i, j] = -J_ij
            herminian_matrix[j, i] = -J_ij

    x = np.random.choice([-1, 1], size=n)
    assert np.array_equal(-2 * x @ J @ x.T, x @ herminian_matrix @ x.T)
    assert np.array_equal(herminian_matrix.T, herminian_matrix)  # wszystkie macierze są rzeczywiste

    new_external_fields = -1 * h
    return herminian_matrix, new_external_fields


## Wyżarzanie równoległe

In [2]:
# własny kernel
import cupy as cp

from typing import Optional
from tqdm import tqdm


def calculate_energy_gpu(J: cp.ndarray, h: cp.ndarray, state: cp.ndarray):
    # Zakładamy, że J jest hermitowska z czynnikiem 1/2
    n, _ = J.shape
    A = cp.multiply(-1/2, J)
    B = cp.matmul(A, state) - h.reshape(n, 1)
    C = cp.multiply(state, B)
    return cp.sum(C, axis=0)



def parrarel_annealing_gpu(J, h, step_size: float, lambda_t_max: float, num_steps: int, num_trajectories: int,
                       schedule: Optional[list] = None, dtype = cp.float32):
    
    n = len(h)
    x = cp.zeros((n, num_trajectories), dtype=dtype)  # stan podstawowy dla H_innit = sum(x**2)
    momentum = cp.zeros((n, num_trajectories), dtype=dtype)
    state = cp.random.choice([dtype(-1.), dtype(1.)], size=(n, num_trajectories)) # losowy stan początkowy
    step_size = dtype(step_size)

    if schedule is None:
        schedule = [dtype(lambda_t_max * (1 - i/(num_steps -1))) for i in range(num_steps)]  # dlaczego nie linspace? chodzi o typy

    kernel = cp.RawModule(path="cuda_kernel/kernel.ptx")
    parrarel_annealing_step = kernel.get_function("parrarel_annealing_step")

    threadsperblock = 256  # Ilość wątków w bloku,
    blockspergrid_x = num_trajectories  # każdy blok zajmuje się trajektorią
    blockspergrid_y = (n + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
    blockspergrid = (blockspergrid_x, blockspergrid_y)

    x_new = cp.empty_like(x)
    momentum_new = cp.empty_like(momentum)
    state_new = cp.empty_like(state)

    for k in tqdm(range(num_steps), desc="wyżarzanie równoległe GPU"):
   
        lambda_t = schedule[k]
        A = cp.matmul(J, state)
        parrarel_annealing_step(blockspergrid, (threadsperblock,), (A, h, x, momentum, 
                                                               lambda_t, step_size, 
                                                               momentum_new, x_new, state_new))
        momentum = momentum_new
        x = x_new
        state = state_new

    return state, calculate_energy_gpu(J, h, state)


In [4]:
import numpy as np
J, h = read_instance(test_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)

J = cp.asarray(J, dtype=cp.float32)
h = cp.asarray(h, dtype=cp.float32)

states, energy = parrarel_annealing_gpu(J, h, step_size=0.01, lambda_t_max=10, num_steps=1000, num_trajectories=64)
print(min(energy))

wyżarzanie równoległe GPU:   0%|          | 0/1000 [00:00<?, ?it/s]

wyżarzanie równoległe GPU: 100%|██████████| 1000/1000 [00:00<00:00, 10176.99it/s]

-469.0


In [ ]:
# E = -12772
# best found -12736 steps 10^4 trajectories 2^10 time approx 43s
# najperw trzeba raz przepuscić by jądro weszło do pamięci podręcznej (można użyć mało kroków)

J, h = read_instance(full_pegasus)
J, h = dwave_conv_to_minus_half_convention(J, h)

J = cp.asarray(J, dtype=cp.float32)
h = cp.asarray(h, dtype=cp.float32)


states, energy = parrarel_annealing_gpu(J, h, step_size=0.01, lambda_t_max=10, num_steps=10000, num_trajectories=2**10)
print(min(energy))

(5400, 5400)


wyżarzanie równoległe GPU: 100%|██████████| 10000/10000 [00:43<00:00, 228.32it/s]


-12736.0


## Bruteforce

In [ ]:
# brute force CUDA

import cupy as cp



    

In [ ]:
# Simulated bifurcation GPU naive

def balistic_simulated_bifurcation(J, h, num_steps, time_step, num_trajectories: int, 
                                   a_0: Optional[float] = None, c_0_scaling: Optional[float] = None):
    if a_0 is None:
        a_0 = 1

    N, _ = J.shape
    mean_J = np.sqrt(np.sum(np.square(J)) / (N * (N - 1)) )
    c_0 = 0.5 / (mean_J * sqrt(N))

    if c_0_scaling is not None:
        c_0 *= c_0_scaling


    a = np.linspace(0, a_0, num=num_steps)

    x = np.zeros((N, num_trajectories))
    y = np.random.uniform(-0.1, 0.1, (N, num_trajectories))

    for t in range(num_steps):
        y += (-1 * (a_0 - a[t]) * x + c_0 * J @ x) * time_step  # x(t)
        x += a_0 * y * time_step # x(t + 1)

        x, y = wall(x, y)

    x = np.sign(x)
    return x, calculate_energy_parrarel(J, h, x, "")

In [ ]:
# własny kernel

from numba import cuda, float64, float32, int8
from nvmath.device import matmul, Dim3
import numpy as np
import cupy as cp

dtype = float32
threadsperblock = 256
block_dim = Dim3(32, 32, 1)
N = 5
num_trajectories = 2





@cuda.jit
def update_x(X, Y, a_0, time_step, X_new, Y_new):
    sX = cuda.shared.array(shape=threadsperblock, dtype=dtype)
    sY = cuda.shared.array(shape=threadsperblock, dtype=dtype)
    
    ti = cuda.threadIdx.x  # pojedyńczy element w kolumnie
    col = cuda.blockIdx.x  # każdy blok zajmuje się jedną kolumną (trajektorią)
    y = cuda.blockIdx.y  # ewentualne dodatkowe bloki na kolumne

    # gdzie "globalnie" jesteśmy w macierzy
    global_row = ti + y * threadsperblock

    temp1 = dtype(0.)
    if global_row < X.shape[0]:
        sX[ti] = X[global_row, col]
        sY[ti] = Y[global_row, col]
    cuda.syncthreads()  # wszystkie wątki widzą dane

    if global_row < X.shape[0]:
        temp1 = sX[ti] + a_0 * sY[ti] * time_step  # krok x

        # "sciana"
        if abs(temp1) > 1:
            X_new[global_row, col] = max(-1, min(1, temp1))
            Y_new[global_row, col] = 0
        else:
            X_new[global_row, col] = temp1
            Y_new[global_row, col] = sY[ti]


J = cuda.to_device(np.random.uniform(-1, 1, (N, N)))
x = cuda.to_device(np.random.uniform(-1, 1, (N, num_trajectories)))
y = cuda.to_device(np.random.uniform(-1, 1, (N, num_trajectories)))
x_new = cuda.device_array_like(x)
y_new = cuda.device_array_like(x)
a_0 = 1
a_t = 0.1
time_step = 0.25
c_0 = 0.25

beta = -1 * (a_0 - a_t)

blockspergrid_x = num_trajectories  # każdy blok zajmuje się trajektorią
blockspergrid_y = (n + threadsperblock - 1) // threadsperblock  # wystarczająca ilość bloków by pomieścić całą kolumnę 
blockspergrid = (blockspergrid_x, blockspergrid_y)
print(MM.files)
# update_y[1, block_dim](J, x, y, c_0, beta, y_new)
# y = y_new
# update_x[blockspergrid, threadsperblock](x, y, a_0, time_step, x_new, y_new)
# print(x_new.copy_to_host())
# print(y_new.copy_to_host())
# print(y.copy_to_host())

# 

In [ ]:
import cupy as cp


module = cp.RawModule(path="cuda_kernel/kernel.ptx")

pa = module.get_function("parrarel_annealing_step")
ker_sum = module.get_function("test_add")

N = 10

x1 = cp.arange(N**2, dtype=cp.float32).reshape(N, N)

x2 = cp.ones((N, N), dtype=cp.float32)

y = cp.zeros((N, N), dtype=cp.float32)

ker_sum((N,), (N,), (x1, x2, y, N**2))   # y = x1 + x2

assert cp.allclose(y, x1 + x2)
print(y)

